# Ablation 3 (Full): Multi-Reranker Comparison

Chạy **tất cả 6 configs** trong 1 session Kaggle.

| # | Config | Fusion | Reranker | Source |
|---|--------|--------|----------|--------|
| 1 | `Rerank-None-Hybrid` | Score merge | ❌ | — |
| 2 | `Rerank-RRF-Hybrid` | RRF | ❌ | — |
| 3 | `Rerank-CrossEncoder-Hybrid` | RRF | mMiniLMv2 | Local GPU |
| 4 | `Rerank-Qwen3Reranker-Hybrid` | RRF | Qwen3-Reranker-0.6B | Local GPU |
| 5 | `Rerank-BGEReranker-Hybrid` | RRF | BGE-Reranker-v2-m3 | Local GPU |
| 6 | `Rerank-JinaReranker-Hybrid` | RRF | Jina-Reranker-v2-base | Local GPU |

**Key features:**
- ✅ Load/unload reranker **tuần tự** (fix OOM)
- ✅ Fix Jina `transformers` compatibility
- ✅ Cache Dense+BM25 từ dataset input (skip ~3.5h)

---

## 1. Install & Imports

In [ ]:
# faiss-gpu cho Kaggle T4, fallback faiss-cpu
import subprocess, sys
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu'], 
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('✅ faiss-gpu installed')
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('⚠️ faiss-gpu failed, using faiss-cpu')

# Upgrade transformers & sentence-transformers for Qwen3 support
!pip install -q rank_bm25 underthesea openai
!pip install -q --upgrade transformers sentence-transformers


In [ ]:
import json, os, sys, time, math, pickle, re, unicodedata, gc, hashlib, shutil
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Sequence
from collections import defaultdict, Counter
from datetime import datetime, timezone
import numpy as np
import torch

# === GPU Detection ===
if torch.cuda.is_available():
    DEVICE = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'🚀 GPU detected: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    DEVICE = 'cpu'
    print('⚠️ No GPU, using CPU (sẽ chậm hơn)')

print(f'Device: {DEVICE}')
print('Setup complete.')

## 2. Configuration

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    FAISS_DIR = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta')
    BM25_BASE_DIR = Path('/kaggle/input/datasets/nguyenlethienlyy/bm25-tokenized/bm25')
    QA_DIR = Path('/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark')
    OUTPUT_BASE = Path('/kaggle/working/evaluation_runs/ablation3_reranker_v2')
    CACHE_DIR = Path('/kaggle/working/retrieval_cache')
    # === Cache từ dataset input (upload pkl files đã chạy trước đó) ===
    CACHE_INPUT_CANDIDATES = [
        Path('/kaggle/input/datasets/kittrntunk/pklcache/cache'),
    ]
else:
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    FAISS_DIR = PROJECT_ROOT / 'data' / 'faiss_index'
    BM25_BASE_DIR = PROJECT_ROOT / 'data' / 'sparse_index'
    QA_DIR = PROJECT_ROOT / 'data' / 'benchmark'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation3_reranker_v2'
    CACHE_DIR = PROJECT_ROOT / 'evaluation_runs' / 'retrieval_cache'
    CACHE_INPUT_CANDIDATES = []

EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
CROSS_ENCODER_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

# --- ALL Configs: baselines + rerankers (chạy tuần tự, KHÔNG load cùng lúc) ---
ALL_CONFIGS = [
    # === Baselines (no reranker model) ===
    {
        'name': 'Rerank-None-Hybrid',
        'description': 'Score merge, no reranking',
        'model_id': None,
        'use_rrf': False, 'use_reranker': False,
        'loader_kwargs': {},
    },
    {
        'name': 'Rerank-RRF-Hybrid',
        'description': 'RRF fusion only',
        'model_id': None,
        'use_rrf': True, 'use_reranker': False,
        'loader_kwargs': {},
    },
    # === Rerankers (load tuần tự) ===
    {
        'name': 'Rerank-CrossEncoder-Hybrid',
        'description': 'RRF + CrossEncoder mMiniLM (baseline)',
        'model_id': CROSS_ENCODER_MODEL,
        'use_rrf': True, 'use_reranker': True,
        'loader_kwargs': {},
    },
    {
        'name': 'Rerank-Qwen3Reranker-Hybrid',
        'description': 'RRF + Qwen3-Reranker-0.6B (local GPU)',
        'model_id': 'Qwen/Qwen3-Reranker-0.6B',
        'use_rrf': True, 'use_reranker': True,
        'loader_kwargs': {'trust_remote_code': True},
    },
    {
        'name': 'Rerank-BGEReranker-Hybrid',
        'description': 'RRF + BGE-Reranker-v2-m3 (local GPU)',
        'model_id': 'BAAI/bge-reranker-v2-m3',
        'use_rrf': True, 'use_reranker': True,
        'loader_kwargs': {'default_activation_function': 'sigmoid'},
    },
    {
        'name': 'Rerank-JinaReranker-Hybrid',
        'description': 'RRF + Jina-Reranker-v2 (local GPU)',
        'model_id': 'jinaai/jina-reranker-v2-base-multilingual',
        'use_rrf': True, 'use_reranker': True,
        'loader_kwargs': {'trust_remote_code': True},
    },
]

TOP_K = 30; TOP_N = 10
CE_CANDIDATE_MULT = 3; RRF_K = 60
SCORE_THRESHOLD = 0.30
TOP_K_EVAL = [1, 3, 5, 10]
ABLATION_LIMIT = None  # Set 5 để smoke test

# --- LLM Generation (Shop AI Key) ---
LLM_BASE_URL = 'https://api.shopaikey.com/v1'
LLM_API_KEY = 'sk-5EAiA6CNDAmXwyewsIMXf4rsZWSkdAGijaEqBFKlWOCC954Z'
LLM_MODEL = 'gpt-4o-mini'
GEN_TEMPERATURE = 0.0; GEN_MAX_TOKENS = 1024; GEN_TIMEOUT = 60

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Local')
print(f'Device: {DEVICE}')
print(f'Cache dir: {CACHE_DIR}')
print(f'\nAll configs ({len(ALL_CONFIGS)}):')
for i, rc in enumerate(ALL_CONFIGS, 1):
    model = rc['model_id'] or 'no reranker'
    print(f'  {i}. {rc["name"]}: {model}')

## 3. Verify Paths & Copy Cache

In [ ]:
def verify_path(path, desc):
    ok = path.exists()
    print(f'  {"✅" if ok else "❌"} {desc}: {path}')
    return ok

print('=== Kiểm tra paths ===')
ok1 = verify_path(FAISS_DIR, 'FAISS')
ok2 = verify_path(BM25_BASE_DIR, 'BM25')
ok3 = verify_path(QA_DIR, 'QA')

BENCHMARK_PATH = None
if ok3:
    for p in ['qa_final.jsonl', '*.jsonl']:
        found = list(QA_DIR.glob(p))
        if found:
            BENCHMARK_PATH = found[0]; break
    print(f'  📝 Benchmark: {BENCHMARK_PATH}')

# === Copy cache từ dataset input nếu có ===
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DENSE_CACHE = CACHE_DIR / 'dense_all_hits.pkl'
BM25_CACHE = CACHE_DIR / 'bm25_all_hits.pkl'

if not DENSE_CACHE.exists() or not BM25_CACHE.exists():
    print('\n=== Tìm cache từ dataset input ===')
    cache_found = False
    for candidate in CACHE_INPUT_CANDIDATES:
        d_src = candidate / 'dense_all_hits.pkl'
        b_src = candidate / 'bm25_all_hits.pkl'
        if d_src.exists() and b_src.exists():
            print(f'  ✅ Found cache at: {candidate}')
            shutil.copy2(d_src, DENSE_CACHE)
            shutil.copy2(b_src, BM25_CACHE)
            print(f'  📦 Copied dense ({d_src.stat().st_size/1024/1024:.1f}MB) + bm25 ({b_src.stat().st_size/1024/1024:.1f}MB)')
            cache_found = True
            break
        else:
            print(f'  ❌ Not at: {candidate}')
    if not cache_found:
        print('  ⚠️ No cache found → sẽ compute từ đầu (chậm ~3.5h cho BM25)')

print(f'\n  💾 Dense cache: {"✅ EXISTS" if DENSE_CACHE.exists() else "❌ not found"}')
print(f'  💾 BM25 cache:  {"✅ EXISTS" if BM25_CACHE.exists() else "❌ not found"}')

if not all([ok1, ok2, ok3, BENCHMARK_PATH]):
    print('\n⚠️ PATHS KHÔNG ĐÚNG!')
else:
    print('\n✅ OK!')

## 4. Schema, Metrics, Generator Utils

In [ ]:
@dataclass(frozen=True)
class SearchHit:
    point_id: str
    score: float
    payload: dict[str, Any]

@dataclass(frozen=True)
class LatencyBreakdown:
    dense_latency_s: float = 0.0
    sparse_latency_s: float = 0.0
    fusion_latency_s: float = 0.0
    cross_encoder_latency_s: float = 0.0
    generation_latency_s: float = 0.0
    total_latency_s: float = 0.0
    def to_dict(self):
        return {k: round(v, 4) for k, v in {
            'dense_latency_s': self.dense_latency_s,
            'sparse_latency_s': self.sparse_latency_s,
            'fusion_latency_s': self.fusion_latency_s,
            'cross_encoder_latency_s': self.cross_encoder_latency_s,
            'generation_latency_s': self.generation_latency_s,
            'total_latency_s': self.total_latency_s,
        }.items()}

print('Schema defined.')

In [ ]:
def recall_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / len(rel) if rel else 0.0
def hit_at_k(ret, rel, k):
    return 1.0 if rel and set(ret[:k]) & rel else 0.0
def mrr_at_k(ret, rel, k):
    if not rel: return 0.0
    for i, c in enumerate(ret[:k], 1):
        if c in rel: return 1.0/i
    return 0.0
def ndcg_at_k(ret, rel, k):
    if not rel: return 0.0
    dcg = sum(1.0/math.log2(i+1) for i, c in enumerate(ret[:k], 1) if c in rel)
    ideal = sum(1.0/math.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg/ideal if ideal else 0.0
def precision_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / k if k else 0.0

_PUNCT_RE = re.compile(r'[^\w\s]', flags=re.UNICODE)
_SPACE_RE = re.compile(r'\s+')
def normalize_text(text):
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFC', text).lower()
    return _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', text)).strip()
def tokenize_text(text):
    n = normalize_text(text)
    return n.split() if n else []
def exact_match(pred, ref):
    return 1.0 if normalize_text(pred) == normalize_text(ref) else 0.0
def token_f1(pred, ref):
    pt, rt = tokenize_text(pred), tokenize_text(ref)
    if not pt and not rt: return 1.0
    if not pt or not rt: return 0.0
    common = sum((Counter(pt) & Counter(rt)).values())
    if common == 0: return 0.0
    p, r = common/len(pt), common/len(rt)
    return 2*p*r/(p+r)
def _lcs_len(a, b):
    prev = [0]*(len(b)+1)
    for ta in a:
        curr = [0]
        for j, tb in enumerate(b, 1):
            curr.append(prev[j-1]+1 if ta == tb else max(prev[j], curr[-1]))
        prev = curr
    return prev[-1]
def rouge_l(pred, ref):
    pt, rt = tokenize_text(pred), tokenize_text(ref)
    if not pt and not rt: return 1.0
    if not pt or not rt: return 0.0
    lcs = _lcs_len(pt, rt)
    p, r = lcs/len(pt), lcs/len(rt)
    return 2*p*r/(p+r) if (p+r) else 0.0
def is_unanswerable_text(text):
    n = normalize_text(text)
    return any(m in n for m in ['không có đủ thông tin','không đủ thông tin','không đủ căn cứ','không tìm thấy','không có thông tin'])
def aggregate_metrics(rows, keys):
    out = {'count': len(rows)}
    for k in keys:
        vals = [float(r[k]) for r in rows if r.get(k) is not None]
        out[k] = sum(vals)/len(vals) if vals else None
    return out
def aggregate_by(rows, field, keys):
    groups = defaultdict(list)
    for r in rows: groups[str(r.get(field) or 'unknown')].append(r)
    return {n: aggregate_metrics(g, keys) for n, g in sorted(groups.items())}

RET_METRIC_KEYS = [f'{n}@{k}' for k in TOP_K_EVAL for n in ['recall','hit','mrr','ndcg','precision']]
GEN_METRIC_KEYS = ['exact_match', 'token_f1', 'rouge_l', 'unanswerable_accuracy']
ALL_METRIC_KEYS = RET_METRIC_KEYS + GEN_METRIC_KEYS
print('Metrics defined.')

In [ ]:
INSUFFICIENT_CONTEXT = 'Không có đủ thông tin trong ngữ cảnh được cung cấp.'
PROMPT_TEMPLATE = """You are a Vietnamese legal retrieval-augmented answering system.
Use only the supplied CONTEXT. Do not invent facts outside it.
If the context is insufficient, answer exactly:
"{insufficient}"

Answer the question directly from the context.

Output contract:
- Return only the final answer; never reveal hidden reasoning or chain-of-thought.
- For answer_type "boolean", the first line must be exactly "Có" or "Không";
  any explanation follows a line beginning "Giải thích:".
- For answer_type "unanswerable", return only the insufficient-context statement.
- Otherwise answer concisely in Vietnamese.
- Cite supporting context with [1], [2], etc. immediately after the claim.
- Do not add a bibliography. Cite only a source that supports the claim.

QUESTION:
{question}

ANSWER_TYPE:
{answer_type}

CONTEXT:
{context}

FINAL ANSWER:"""

_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE)
_CITE_RE = re.compile(r'\[\s*(\d+)\s*\]')

def format_chunks_as_context(hits):
    blocks = []
    for i, hit in enumerate(hits, 1):
        p = hit.payload
        lines = [f'[SOURCE {i}]']
        for label, key in [('Title','title'),('Article','article_number'),('Section','citation_anchor'),('Chunk ID','chunk_id')]:
            v = p.get(key)
            if v: lines.append(f'{label}: {v}')
        lines.extend(['Content:', str(p.get('chunk_text') or p.get('text') or '')[:4000]])
        blocks.append('\n'.join(lines))
    return '\n\n'.join(blocks)

def build_prompt(question, answer_type, hits):
    return PROMPT_TEMPLATE.format(insufficient=INSUFFICIENT_CONTEXT, question=question.strip(),
                                  answer_type=(answer_type or '').strip(), context=format_chunks_as_context(hits))
def parse_answer(raw_text):
    return _THINK_RE.sub('', raw_text or '').strip()
def count_citations(text):
    return len(set(_CITE_RE.findall(text or '')))

print('Generator utils defined.')

In [ ]:
from openai import OpenAI
gen_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def call_llm(prompt, *, temperature=GEN_TEMPERATURE, max_tokens=GEN_MAX_TOKENS):
    resp = gen_client.chat.completions.create(
        model=LLM_MODEL, messages=[{'role':'user','content':prompt}],
        temperature=temperature, max_tokens=max_tokens, timeout=GEN_TIMEOUT)
    return (resp.choices[0].message.content or '').strip()

print('Testing LLM...')
print(f'  Response: {call_llm("1+1 bằng mấy?")[:100]}')
print('✅ Generator ready.')

## 5. Load QA Benchmark

In [ ]:
print(f'Loading benchmark from {BENCHMARK_PATH} ...')
qa_data = []
with open(BENCHMARK_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: qa_data.append(json.loads(line))

eval_qa, skip_no_gt = [], 0
for qa in qa_data:
    gt = qa.get('ground_truth') or {}
    gt_chunks = {str(c) for c in gt.get('chunk_ids') or [] if c}
    at = str(qa.get('answer_type') or '').lower()
    cat = str(qa.get('category') or '').lower()
    is_unanswerable = (at == 'unanswerable' or cat == 'unanswerable')
    if not gt_chunks and not is_unanswerable:
        skip_no_gt += 1; continue
    eval_qa.append((qa, gt_chunks, is_unanswerable))

if ABLATION_LIMIT: eval_qa = eval_qa[:ABLATION_LIMIT]
all_questions = [str(qa.get('question') or '') for qa, _, _ in eval_qa]
n_unans = sum(1 for _,_,u in eval_qa if u)
print(f'Eval: {len(eval_qa)} total ({len(eval_qa)-n_unans} answerable + {n_unans} unanswerable)')

## 6. Load/Compute Dense + BM25 (with cache)

Nếu đã upload pkl cache từ dataset → **load ngay, skip hoàn toàn**

In [ ]:
def _hits_to_dicts(hits_list):
    return [[(h.point_id, h.score, h.payload) for h in hits] for hits in hits_list]
def _dicts_to_hits(data):
    return [[SearchHit(pid, sc, pay) for pid, sc, pay in hits] for hits in data]

# Try loading cache
if DENSE_CACHE.exists() and BM25_CACHE.exists():
    print('💾 Loading from cache...')
    t0 = time.perf_counter()
    with open(DENSE_CACHE, 'rb') as f:
        cache_dense = pickle.load(f)
    dense_all_hits = _dicts_to_hits(cache_dense['hits'])
    dense_all_latencies = cache_dense['latencies']
    print(f'  ✅ Dense: {len(dense_all_hits)} queries ({time.perf_counter()-t0:.1f}s)')

    t0 = time.perf_counter()
    with open(BM25_CACHE, 'rb') as f:
        cache_bm25 = pickle.load(f)
    bm25_all_hits = _dicts_to_hits(cache_bm25['hits'])
    print(f'  ✅ BM25:  {len(bm25_all_hits)} queries ({time.perf_counter()-t0:.1f}s)')

    if len(dense_all_hits) != len(eval_qa) or len(bm25_all_hits) != len(eval_qa):
        print(f'  ⚠️ Cache size mismatch! Cache={len(dense_all_hits)}, Eval={len(eval_qa)}')
        print(f'  → Will re-compute.')
        dense_all_hits = None
    else:
        print(f'\n✅ Cache valid! Skip retrieval → straight to reranking.')
else:
    dense_all_hits = None
    print('❌ No cache → will compute Dense + BM25 from scratch.')

In [ ]:
# === DENSE: compute if no cache ===
if dense_all_hits is None:
    import faiss
    from sentence_transformers import SentenceTransformer

    print('Loading FAISS index...')
    t0 = time.perf_counter()
    faiss_index = faiss.read_index(str(FAISS_DIR / 'index.faiss'))
    
    if DEVICE == 'cuda':
        try:
            res = faiss.StandardGpuResources()
            faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_index)
            print(f'  🚀 FAISS on GPU ({faiss_index.ntotal:,} vectors, {time.perf_counter()-t0:.1f}s)')
        except Exception as e:
            print(f'  ⚠️ FAISS GPU failed ({e}), using CPU')
            print(f'  {faiss_index.ntotal:,} vectors ({time.perf_counter()-t0:.1f}s)')
    else:
        print(f'  {faiss_index.ntotal:,} vectors ({time.perf_counter()-t0:.1f}s)')

    print('Loading payloads...')
    t0 = time.perf_counter()
    dense_payloads = {}
    with open(FAISS_DIR / 'payloads.jsonl', 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if line: dense_payloads[i] = json.loads(line)
    print(f'  {len(dense_payloads):,} payloads ({time.perf_counter()-t0:.1f}s)')

    id_map_path = FAISS_DIR / 'id_map.json'
    if id_map_path.exists():
        with open(id_map_path, 'r', encoding='utf-8') as f: raw = json.load(f)
        dense_id_map = {int(v): str(k) for k, v in raw.items()}
    else:
        dense_id_map = {i: str(dense_payloads.get(i, {}).get('chunk_id', i)) for i in dense_payloads}

    print(f'Loading {EMBEDDING_MODEL} on {DEVICE}...')
    t0 = time.perf_counter()
    embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
    print(f'  🚀 Loaded on {DEVICE} in {time.perf_counter()-t0:.1f}s')

    def dense_search(query, *, top_k=30, score_threshold=0.0):
        vec = embedder.encode(['query: ' + query], normalize_embeddings=True)
        qv = np.array(vec, dtype=np.float32)
        limit = min(top_k * 3, faiss_index.ntotal)
        scores, indices = faiss_index.search(qv, limit)
        hits = []
        for sc, idx in zip(scores[0], indices[0]):
            if idx < 0 or float(sc) < score_threshold: continue
            hits.append(SearchHit(point_id=dense_id_map.get(int(idx), str(idx)),
                                  score=float(sc), payload=dense_payloads.get(int(idx), {})))
            if len(hits) >= top_k: break
        return hits

    print(f'\nPre-computing dense for {len(eval_qa)} queries on {DEVICE}...')
    dense_all_hits, dense_all_latencies = [], []
    dense_t0 = time.perf_counter()
    for qi in range(len(eval_qa)):
        t0 = time.perf_counter()
        hits = dense_search(all_questions[qi], top_k=TOP_K, score_threshold=SCORE_THRESHOLD)
        dense_all_hits.append(hits)
        dense_all_latencies.append(time.perf_counter() - t0)
        if (qi+1) % 100 == 0:
            print(f'  {qi+1}/{len(eval_qa)} ({time.perf_counter()-dense_t0:.0f}s)')
    print(f'✅ Dense done in {time.perf_counter()-dense_t0:.0f}s')

    with open(DENSE_CACHE, 'wb') as f:
        pickle.dump({'hits': _hits_to_dicts(dense_all_hits), 'latencies': dense_all_latencies}, f)
    print(f'💾 Dense cache saved ({DENSE_CACHE.stat().st_size/1024/1024:.1f} MB)')

    del faiss_index, dense_payloads, dense_id_map, embedder
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
else:
    print('⏭️ Dense: using cache')

In [ ]:
# === BM25: compute if no cache ===
if not BM25_CACHE.exists():
    def simple_tokenize(text):
        text = unicodedata.normalize('NFC', text).lower()
        text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
        return [t for t in text.split() if len(t) > 1]
    try:
        from underthesea import word_tokenize as _ws
        def bm25_tokenize(text): return simple_tokenize(_ws(text, format='text'))
        print('✅ underthesea tokenizer')
    except ImportError:
        bm25_tokenize = simple_tokenize
        print('⚠️ simple tokenizer')

    all_tokenized = [bm25_tokenize(q) for q in all_questions]
    shard_dirs = sorted([d for d in BM25_BASE_DIR.iterdir()
                         if d.is_dir() and d.name.startswith('shard_')])
    print(f'Found {len(shard_dirs)} shards, {len(all_tokenized)} queries')

    bm25_all_hits = [[] for _ in range(len(eval_qa))]
    bm25_t0 = time.perf_counter()

    for shard_dir in shard_dirs:
        st0 = time.perf_counter()
        idx_p, meta_p = shard_dir / 'bm25_index.pkl', shard_dir / 'bm25_metadata.pkl'
        if not idx_p.exists() or not meta_p.exists():
            print(f'  ⚠️ Skip {shard_dir.name}'); continue
        with idx_p.open('rb') as f: bm25 = pickle.load(f)
        with meta_p.open('rb') as f: meta = pickle.load(f)
        chunk_ids = meta['chunk_ids']
        shard_pays = meta.get('payloads', [{}]*len(chunk_ids))
        del meta
        load_t = time.perf_counter() - st0

        s_t0 = time.perf_counter()
        for qi, tok_q in enumerate(all_tokenized):
            scores = bm25.get_scores(tok_q)
            top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:TOP_K]
            for idx in top_idx:
                sc = float(scores[idx])
                if sc <= 0: break
                cid = chunk_ids[idx]
                pay = shard_pays[idx] if idx < len(shard_pays) and isinstance(shard_pays[idx], dict) else {'chunk_id': cid}
                bm25_all_hits[qi].append(SearchHit(point_id=cid, score=sc, payload=pay))
        print(f'  ✅ {shard_dir.name}: {len(chunk_ids):,} docs | load={load_t:.1f}s | search={time.perf_counter()-s_t0:.1f}s')
        del bm25, chunk_ids, shard_pays; gc.collect()

    for qi in range(len(eval_qa)):
        bm25_all_hits[qi].sort(key=lambda h: h.score, reverse=True)
        seen, deduped = set(), []
        for h in bm25_all_hits[qi]:
            cid = str(h.payload.get('chunk_id') or h.point_id)
            if cid not in seen: seen.add(cid); deduped.append(h)
            if len(deduped) >= TOP_K: break
        bm25_all_hits[qi] = deduped

    print(f'\n✅ BM25 done in {time.perf_counter()-bm25_t0:.0f}s')
    with open(BM25_CACHE, 'wb') as f:
        pickle.dump({'hits': _hits_to_dicts(bm25_all_hits)}, f)
    print(f'💾 BM25 cache saved ({BM25_CACHE.stat().st_size/1024/1024:.1f} MB)')
else:
    print('⏭️ BM25: using cache')

print(f'\n📦 Ready: dense[{len(dense_all_hits)}] + bm25[{len(bm25_all_hits)}]')

## 7. Fusion Functions

In [ ]:
def hybrid_merge_score(dense_hits, bm25_hits, top_n=10):
    d_max = max((h.score for h in dense_hits), default=1.0) or 1.0
    b_max = max((h.score for h in bm25_hits), default=1.0) or 1.0
    combined = {}
    for h in dense_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        n = h.score / d_max
        if cid not in combined or n > combined[cid].score:
            combined[cid] = SearchHit(h.point_id, n, h.payload)
    for h in bm25_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        n = h.score / b_max
        if cid not in combined or n > combined[cid].score:
            combined[cid] = SearchHit(h.point_id, n, h.payload)
    return sorted(combined.values(), key=lambda h: h.score, reverse=True)[:top_n]

def rrf_fusion(dense_hits, bm25_hits, *, k=60, top_n=10):
    rrf_scores, best_hit = {}, {}
    for rank, h in enumerate(dense_hits, 1):
        cid = str(h.payload.get('chunk_id') or h.point_id)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(k+rank)
        if cid not in best_hit: best_hit[cid] = h
    for rank, h in enumerate(bm25_hits, 1):
        cid = str(h.payload.get('chunk_id') or h.point_id)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(k+rank)
        if cid not in best_hit: best_hit[cid] = h
    sorted_ids = sorted(rrf_scores, key=lambda c: rrf_scores[c], reverse=True)
    return [SearchHit(best_hit[c].point_id, rrf_scores[c], best_hit[c].payload) for c in sorted_ids[:top_n]]

print('Fusion functions defined.')

## 8. Reranker Loader (tuần tự — fix OOM)

Chỉ load **1 reranker tại 1 thời điểm**.  
Sau khi chạy xong → `del model` + `torch.cuda.empty_cache()` → load model tiếp theo.

In [ ]:
import gc, sys, time, torch
from sentence_transformers import CrossEncoder

# === Monkey-patch for Jina-v2 compatibility with Transformers 4.48+ ===
try:
    import transformers.models.xlm_roberta.modeling_xlm_roberta as xlm_mod
    if not hasattr(xlm_mod, 'create_position_ids_from_input_ids'):
        def _create_pos_ids(input_ids, padding_idx, past_key_values_length=0):
            mask = input_ids.ne(padding_idx).int()
            incremental_indices = (torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask
            return incremental_indices.long() + padding_idx
        setattr(xlm_mod, 'create_position_ids_from_input_ids', _create_pos_ids)
        print('✅ Monkey-patch applied for Jina XLM-RoBERTa compatibility.')
except Exception as patch_e:
    print(f'⚠️ Jina monkey-patch note: {patch_e}')

class AutoRerankerWrapper:
    """FP16 Optimized Wrapper for Qwen3 / CausalLM / SequenceClassification models to prevent CUDA OOM."""
    def __init__(self, model_id, device='cuda'):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True
        ).to(device)
        self.model.eval()

    def predict(self, pairs, batch_size=8):
        scores = []
        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i:i+batch_size]
            inputs = self.tokenizer(
                batch_pairs, padding=True, truncation=True, max_length=1024, return_tensors='pt'
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                logits = outputs.logits
                if logits.dim() == 2 and logits.shape[1] > 1:
                    batch_scores = torch.softmax(logits, dim=-1)[:, -1].cpu().numpy().tolist()
                else:
                    batch_scores = logits.squeeze(-1).cpu().numpy().tolist()
                scores.extend(batch_scores if isinstance(batch_scores, list) else [float(batch_scores)])
            del inputs, outputs
            if self.device == 'cuda' and i % 64 == 0:
                torch.cuda.empty_cache()
        return scores

def load_reranker(config):
    """Load a single reranker model onto GPU."""
    model_id = config['model_id']
    kwargs = {}
    
    loader_kwargs = config.get('loader_kwargs', {})
    if loader_kwargs.get('default_activation_function') == 'sigmoid':
        kwargs['default_activation_function'] = torch.nn.Sigmoid()
    if loader_kwargs.get('trust_remote_code'):
        kwargs['trust_remote_code'] = True
    
    print(f'  Loading {model_id} on {DEVICE}...')
    t0 = time.perf_counter()
    try:
        model = CrossEncoder(model_id, device=DEVICE, **kwargs)
    except Exception as e:
        print(f'  ⚠️ CrossEncoder load failed ({e}), using AutoRerankerWrapper fallback...')
        model = AutoRerankerWrapper(model_id, device=DEVICE)

    print(f'  🚀 Loaded in {time.perf_counter()-t0:.1f}s')
    
    if DEVICE == 'cuda':
        used = torch.cuda.memory_allocated() / 1024**3
        print(f'  📊 VRAM used: {used:.1f} GB')
    
    return model

def unload_reranker(model):
    """Unload reranker from GPU to free VRAM."""
    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1024**3
        print(f'  🗑️ Model unloaded. VRAM used: {used:.1f} GB')

def rerank_with_model(model, query, hits, *, top_n=10):
    """Rerank hits using a loaded CrossEncoder model or AutoRerankerWrapper."""
    if not hits: return [], 0.0
    t0 = time.perf_counter()
    pairs = [(query, str(h.payload.get('chunk_text') or '')[:1500]) for h in hits]
    scores = model.predict(pairs)
    scored = [SearchHit(h.point_id, float(s), h.payload) for h, s in zip(hits, scores)]
    scored.sort(key=lambda h: h.score, reverse=True)
    return scored[:top_n], time.perf_counter() - t0

print('Testing load/unload cycle...')
if DEVICE == 'cuda':
    print(f'  VRAM before: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print('✅ Reranker loader ready.')

## 9. Pre-compute RRF Fusion (shared across all configs)

In [ ]:
# Pre-compute fusion candidates for all queries
print(f'Pre-computing fusion for {len(eval_qa)} queries...')
n_cand = TOP_N * CE_CANDIDATE_MULT  # 30 candidates for reranking
t0 = time.perf_counter()

rrf_fused_all = []
score_merge_all = []
for qi in range(len(eval_qa)):
    rrf_fused_all.append(rrf_fusion(dense_all_hits[qi], bm25_all_hits[qi], k=RRF_K, top_n=n_cand))
    score_merge_all.append(hybrid_merge_score(dense_all_hits[qi], bm25_all_hits[qi], top_n=TOP_N))

print(f'✅ Fusion done in {time.perf_counter()-t0:.1f}s')
print(f'   RRF avg candidates: {np.mean([len(f) for f in rrf_fused_all]):.1f}')
print(f'   Score-merge avg: {np.mean([len(f) for f in score_merge_all]):.1f}')

## 10. Run Ablation (tuần tự — 1 model tại 1 thời điểm)

In [ ]:
all_results = {}
grand_t0 = time.perf_counter()

for rc_idx, rc in enumerate(ALL_CONFIGS):
    cfg_name = rc['name']
    model_id = rc['model_id']
    use_rrf = rc['use_rrf']
    use_reranker = rc['use_reranker']
    
    print(f'\n{"="*70}')
    print(f'[{rc_idx+1}/{len(ALL_CONFIGS)}] {cfg_name}')
    print(f'  Model: {model_id or "none"}')
    print(f'  Description: {rc["description"]}')
    print(f'{"="*70}')
    
    # === LOAD reranker (only if needed) ===
    reranker_model = None
    if use_reranker:
        try:
            reranker_model = load_reranker(rc)
        except Exception as e:
            print(f'  ❌ FAILED to load {model_id}: {e}')
            print(f'  → SKIPPING this config')
            continue
    
    # === RUN evaluation ===
    cases, latencies, gen_errors = [], [], 0
    run_t0 = time.perf_counter()

    for qi, (qa, gt_chunks, is_unanswerable) in enumerate(eval_qa):
        question = all_questions[qi]
        qa_id = str(qa.get('qa_id') or qa.get('id') or f'qa_{qi+1}')
        answer_type = str(qa.get('answer_type') or '')
        ref_answer = str(qa.get('reference_answer') or qa.get('answer') or '')
        try:
            t0 = time.perf_counter()
            d_lat = dense_all_latencies[qi] if 'dense_all_latencies' in dir() else 0.0

            # --- Fusion ---
            f_t0 = time.perf_counter()
            if not use_rrf:
                # Score merge (None config)
                fused = score_merge_all[qi]
            elif use_reranker:
                # RRF with more candidates for reranking
                fused = rrf_fused_all[qi]
            else:
                # RRF only (no reranker)
                fused = rrf_fusion(dense_all_hits[qi], bm25_all_hits[qi], k=RRF_K, top_n=TOP_N)
            f_lat = time.perf_counter() - f_t0

            # --- Reranking ---
            ce_lat = 0.0
            if use_reranker and reranker_model is not None:
                fused, ce_lat = rerank_with_model(reranker_model, question, fused, top_n=TOP_N)
            else:
                fused = fused[:TOP_N]

            hits = fused
            ret_lat = LatencyBreakdown(dense_latency_s=d_lat, fusion_latency_s=f_lat,
                                       cross_encoder_latency_s=ce_lat, total_latency_s=time.perf_counter()-t0)
            ret_ids = [str(h.payload.get('chunk_id') or h.point_id) for h in hits]

            # --- Generation ---
            gen_lat, predicted, citation_count, gen_error = 0.0, '', 0, None
            if hits:
                try:
                    gen_t0 = time.perf_counter()
                    predicted = parse_answer(call_llm(build_prompt(question, answer_type, hits)))
                    gen_lat = time.perf_counter() - gen_t0
                    citation_count = count_citations(predicted)
                except Exception as e:
                    gen_error = str(e); gen_errors += 1
                    if gen_errors <= 3: print(f'  ⚠️ Gen error {qa_id}: {str(e)[:100]}')
            else:
                predicted = INSUFFICIENT_CONTEXT

            row = {'qa_id': qa_id, 'question': question, 'category': qa.get('category'),
                   'difficulty': qa.get('difficulty'), 'answer_type': answer_type,
                   'is_unanswerable': is_unanswerable, 'reference_answer': ref_answer,
                   'predicted_answer': predicted, 'ground_truth_chunk_ids': sorted(gt_chunks),
                   'retrieved_chunk_ids': ret_ids, 'num_retrieved': len(ret_ids),
                   'citation_count': citation_count, 'generation_error': gen_error}

            if gt_chunks:
                for k in TOP_K_EVAL:
                    row[f'recall@{k}'] = recall_at_k(ret_ids, gt_chunks, k)
                    row[f'hit@{k}'] = hit_at_k(ret_ids, gt_chunks, k)
                    row[f'mrr@{k}'] = mrr_at_k(ret_ids, gt_chunks, k)
                    row[f'ndcg@{k}'] = ndcg_at_k(ret_ids, gt_chunks, k)
                    row[f'precision@{k}'] = precision_at_k(ret_ids, gt_chunks, k)

            if is_unanswerable:
                row['exact_match'] = row['token_f1'] = row['rouge_l'] = None
                row['unanswerable_accuracy'] = 1.0 if is_unanswerable_text(predicted) else 0.0
            else:
                if predicted and not gen_error:
                    pe = predicted.split('\n')[0].strip() if answer_type=='boolean' else predicted
                    re_ = ref_answer.split('\n')[0].strip() if answer_type=='boolean' else ref_answer
                    row['exact_match'] = exact_match(pe, re_)
                    row['token_f1'] = token_f1(predicted, ref_answer)
                    row['rouge_l'] = rouge_l(predicted, ref_answer)
                else:
                    row['exact_match'] = row['token_f1'] = row['rouge_l'] = 0.0
                row['unanswerable_accuracy'] = 1.0 if not is_unanswerable_text(predicted) else 0.0

            cases.append(row)
            latencies.append(LatencyBreakdown(dense_latency_s=ret_lat.dense_latency_s,
                fusion_latency_s=ret_lat.fusion_latency_s, cross_encoder_latency_s=ret_lat.cross_encoder_latency_s,
                generation_latency_s=gen_lat, total_latency_s=ret_lat.total_latency_s+gen_lat).to_dict())
        except Exception as e:
            print(f'  ❌ FATAL {qa_id}: {e}')
            cases.append({'qa_id': qa_id, 'error': str(e)})
            latencies.append(LatencyBreakdown().to_dict())

        if (qi+1) % 25 == 0:
            el = time.perf_counter()-run_t0; eta = (el/(qi+1))*(len(eval_qa)-qi-1)
            print(f'  {qi+1}/{len(eval_qa)} ({el:.0f}s, ~{eta:.0f}s left)')

    dur = time.perf_counter()-run_t0
    valid = [c for c in cases if 'error' not in c]
    
    summary = {
        'config_name': cfg_name, 
        'config': {'description': rc['description'], 'model_id': model_id or 'none', 'use_rrf': use_rrf, 'use_reranker': use_reranker},
        'counts': {'total': len(cases), 'evaluated': len(valid), 'errors': len(cases)-len(valid), 'gen_errors': gen_errors},
        'overall': aggregate_metrics(valid, ALL_METRIC_KEYS),
        'by_category': aggregate_by(valid, 'category', ALL_METRIC_KEYS),
        'by_difficulty': aggregate_by(valid, 'difficulty', ALL_METRIC_KEYS),
        'by_answer_type': aggregate_by(valid, 'answer_type', ALL_METRIC_KEYS),
        'latency': {
            'total_run_time_s': round(dur, 2),
            'avg': {k: round(np.mean([l[k] for l in latencies]), 4) for k in latencies[0]} if latencies else {},
            'median': {k: round(float(np.median([l[k] for l in latencies])), 4) for k in latencies[0]} if latencies else {},
        },
    }
    print(f'\n  ✅ {cfg_name}: {len(valid)}/{len(cases)} cases, {dur:.0f}s')
    for m in ['recall@10','mrr@10','ndcg@10','token_f1','rouge_l','exact_match']:
        v = summary['overall'].get(m)
        print(f'     {m}: {v:.4f}' if v is not None else f'     {m}: N/A')
    all_results[cfg_name] = (cases, latencies, summary)
    
    # === UNLOAD reranker to free VRAM ===
    if reranker_model is not None:
        print(f'\n  Unloading {model_id}...')
        unload_reranker(reranker_model)
        reranker_model = None

grand_total = time.perf_counter()-grand_t0
print(f'\n{"="*70}\n✅ All done! {grand_total:.0f}s ({grand_total/60:.1f}min)\n{"="*70}')

## 11. Save Results

In [ ]:
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
for cfg_name, (cases, latencies, summary) in all_results.items():
    cfg_dir = OUTPUT_BASE / cfg_name; cfg_dir.mkdir(parents=True, exist_ok=True)
    with open(cfg_dir/'retrieval_cases.jsonl','w',encoding='utf-8') as f:
        for c in cases: json.dump(c,f,ensure_ascii=False,default=str); f.write('\n')
    with open(cfg_dir/'retrieval_metrics.json','w',encoding='utf-8') as f:
        json.dump(summary,f,indent=2,ensure_ascii=False,default=str)
    with open(cfg_dir/'latency.json','w',encoding='utf-8') as f:
        json.dump(summary['latency'],f,indent=2,ensure_ascii=False)
    with open(cfg_dir/'manifest.json','w',encoding='utf-8') as f:
        json.dump({'config':cfg_name,'timestamp':datetime.now(timezone.utc).isoformat(),
                   'eval_count':len(cases),'embedding_model':EMBEDDING_MODEL,
                   'reranker_model': summary['config'].get('model_id', 'none'),
                   'generator_model':LLM_MODEL,
                   'top_k':TOP_K,'top_n':TOP_N,'rrf_k':RRF_K,'ce_candidate_mult':CE_CANDIDATE_MULT,
                   'device': DEVICE},
                  f,indent=2,ensure_ascii=False)
    print(f'  ✅ {cfg_name}')

combined = {'owner':'Ablation Multi-Reranker v2','ablation':'Ablation 3 Full: Multi-Reranker',
    'timestamp':datetime.now(timezone.utc).isoformat(),'device': DEVICE,
    'settings':{'embedding_model':EMBEDDING_MODEL,
                'reranker_models': {rc['name']: (rc['model_id'] or 'none') for rc in ALL_CONFIGS},
                'generator_model':LLM_MODEL,'top_k':TOP_K,'top_n':TOP_N,'rrf_k':RRF_K,
                'ce_candidate_mult':CE_CANDIDATE_MULT,'eval_count':len(eval_qa)},
    'configs':{n:{'overall':s['overall'],'latency':s['latency'],'by_category':s['by_category'],
                  'by_answer_type':s['by_answer_type']} for n,(_,_,s) in all_results.items()}}
with open(OUTPUT_BASE/'summary.json','w',encoding='utf-8') as f:
    json.dump(combined,f,indent=2,ensure_ascii=False,default=str)
print(f'\n✅ summary.json saved')

## 12. Comparison Table

In [ ]:
import csv
csv_path = OUTPUT_BASE / 'comparison.csv'
fieldnames = ['Config'] + RET_METRIC_KEYS + GEN_METRIC_KEYS + ['avg_reranker_latency','avg_gen_latency','avg_total_latency']
rows_csv = []
for name,(_,_,s) in all_results.items():
    row = {'Config': name}
    for k in RET_METRIC_KEYS+GEN_METRIC_KEYS:
        v = s['overall'].get(k); row[k] = round(v,4) if v is not None else ''
    row['avg_reranker_latency'] = s['latency']['avg'].get('cross_encoder_latency_s',0)
    row['avg_gen_latency'] = s['latency']['avg'].get('generation_latency_s',0)
    row['avg_total_latency'] = s['latency']['avg'].get('total_latency_s',0)
    rows_csv.append(row)
with open(csv_path,'w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f,fieldnames=fieldnames); w.writeheader(); w.writerows(rows_csv)

print('='*120)
print('ABLATION 3 EXTENDED v2: MULTI-RERANKER COMPARISON')
print('='*120)
print(f'{"Config":<40} {"Eval":>5} {"R@10":>6} {"MRR@10":>7} {"nDCG@10":>8} {"F1":>6} {"RL":>6} {"EM":>6} {"Rerank_lat":>10} {"Tot_lat":>8}')
print('-'*120)
for name,(_,_,s) in all_results.items():
    o,lat = s['overall'], s['latency']['avg']
    n_eval = s['counts']['evaluated']
    print(f'{name:<40} {n_eval:>5} {(o.get("recall@10",0) or 0):>6.4f} {(o.get("mrr@10",0) or 0):>7.4f} '
          f'{(o.get("ndcg@10",0) or 0):>8.4f} {(o.get("token_f1",0) or 0):>6.4f} '
          f'{(o.get("rouge_l",0) or 0):>6.4f} {(o.get("exact_match",0) or 0):>6.4f} '
          f'{lat.get("cross_encoder_latency_s",0):>10.4f} {lat.get("total_latency_s",0):>8.4f}')
print('='*120)
print(f'\n📊 CSV saved: {csv_path}')

## 13. Visualization

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

cn = list(all_results.keys())
short_names = {
    'Rerank-None-Hybrid': 'None',
    'Rerank-RRF-Hybrid': 'RRF',
    'Rerank-CrossEncoder-Hybrid': 'CE',
    'Rerank-Qwen3Reranker-Hybrid': 'Qwen3',
    'Rerank-BGEReranker-Hybrid': 'BGE',
    'Rerank-JinaReranker-Hybrid': 'Jina',
}
sn = [short_names.get(n, n[:12]) for n in cn]
colors = ['#4A90D9','#E8833A','#50B86C','#D94A6B','#9B59B6','#F39C12'][:len(cn)]
gm = lambda n,m: all_results[n][2]['overall'].get(m,0) or 0

fig, axes = plt.subplots(2,3,figsize=(22,12))
fig.suptitle('Ablation 3 Full: Multi-Reranker Comparison (6 configs)', fontsize=16, fontweight='bold')

# Row 1: R@k, MRR@k, nDCG@k curves
for ax,metric in zip([axes[0,0],axes[0,1],axes[0,2]], ['recall','mrr','ndcg']):
    for i,n in enumerate(cn):
        ax.plot(TOP_K_EVAL,[gm(n,f'{metric}@{k}') for k in TOP_K_EVAL],'o-',label=sn[i],color=colors[i],lw=2,ms=7)
    ax.set_title(f'{metric.upper()}@k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3); ax.set_xticks(TOP_K_EVAL)

# Row 2: Generation metrics
ax=axes[1,0]; x=np.arange(3); w=0.12
for i,n in enumerate(cn):
    ax.bar(x+i*w,[gm(n,m) for m in ['exact_match','token_f1','rouge_l']],w,label=sn[i],color=colors[i])
ax.set_title('Generation Metrics'); ax.set_xticks(x+w*(len(cn)-1)/2); ax.set_xticklabels(['EM','F1','RL']); ax.legend(fontsize=7)

# Latency breakdown
ax=axes[1,1]; bottom=np.zeros(len(cn))
for lk,ll,lc in zip(['dense_latency_s','fusion_latency_s','cross_encoder_latency_s','generation_latency_s'],
                     ['Dense','Fusion','Reranker','Gen'],['#4A90D9','#E8833A','#D94A6B','#50B86C']):
    vs=[all_results[n][2]['latency']['avg'].get(lk,0) for n in cn]
    ax.bar(sn,vs,bottom=bottom,label=ll,color=lc); bottom+=np.array(vs)
ax.set_title('Latency Breakdown (avg/query)'); ax.legend(fontsize=7)

# Key metrics summary
ax=axes[1,2]; x=np.arange(5); w=0.12
for i,n in enumerate(cn):
    ax.bar(x+i*w,[gm(n,m) for m in ['recall@10','mrr@10','ndcg@10','token_f1','rouge_l']],w,label=sn[i],color=colors[i])
ax.set_title('Key Metrics Summary'); ax.set_xticks(x+w*(len(cn)-1)/2); ax.set_xticklabels(['R@10','MRR','nDCG','F1','RL']); ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_BASE/'charts.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ Charts saved')

## 14. Detailed Analysis

In [ ]:
for breakdown_name, breakdown_key in [('PER-CATEGORY','by_category'),('PER-ANSWER-TYPE','by_answer_type')]:
    print(f'\n{"="*120}\n{breakdown_name}\n{"="*120}')
    groups = set()
    for _,(_,_,s) in all_results.items(): groups.update(s[breakdown_key].keys())
    for g in sorted(groups):
        print(f'\n--- {g.upper()} ---')
        print(f'{"Config":<40} {"R@10":>6} {"MRR@10":>7} {"nDCG@10":>8} {"F1":>6} {"RL":>6} {"Count":>6}')
        for name,(_,_,s) in all_results.items():
            d = s[breakdown_key].get(g, {})
            if not d: continue
            print(f'{name:<40} {(d.get("recall@10",0) or 0):>6.4f} {(d.get("mrr@10",0) or 0):>7.4f} '
                  f'{(d.get("ndcg@10",0) or 0):>8.4f} {(d.get("token_f1",0) or 0):>6.4f} '
                  f'{(d.get("rouge_l",0) or 0):>6.4f} {d.get("count",0):>6}')

In [ ]:
print(f'\n{"="*70}\nABLATION 3 FULL: MULTI-RERANKER — FINAL SUMMARY\n{"="*70}')
print(f'Runtime: {grand_total:.0f}s ({grand_total/60:.1f}min) | Queries: {len(eval_qa)} | Configs: {len(all_results)}')
print(f'Device: {DEVICE}')
print(f'Embedding: {EMBEDDING_MODEL}')
print(f'Generator: {LLM_MODEL}')

print(f'\nConfigs:')
for name,(_,_,s) in all_results.items():
    model = s['config'].get('model_id', '?')
    n_ok = s['counts']['evaluated']
    n_err = s['counts']['errors']
    print(f'  • {name}: {model} ({n_ok}/{n_ok+n_err} OK, {n_err} errors)')

print(f'\n--- Best by metric ---')
for mn,mk in [('Recall@10','recall@10'),('MRR@10','mrr@10'),('nDCG@10','ndcg@10'),
              ('Token F1','token_f1'),('ROUGE-L','rouge_l'),('Exact Match','exact_match')]:
    bn = max(all_results, key=lambda n: all_results[n][2]['overall'].get(mk,0) or 0)
    bv = all_results[bn][2]['overall'].get(mk,0) or 0
    print(f'  {mn:<15}: {bn} ({bv:.4f})')

print(f'\n--- Reranker Latency (avg per query) ---')
for name,(_,_,s) in all_results.items():
    lat = s['latency']['avg'].get('cross_encoder_latency_s', 0)
    print(f'  {name:<40}: {lat:.4f}s')

print(f'\n✅ Output: {OUTPUT_BASE}')